# Strands Agents with Bedrock AgentCore Memory — FSI Edition

This lab demonstrates persistent memory for AI agents — enabling them to remember client preferences, risk profiles, and past interactions across sessions.

## Overview

In this lab, you will:
- Create an AgentCore Memory instance with multiple strategies
- Store FSI client conversations (risk appetite, migration plans)
- Retrieve memories across sessions
- Build a memory-enabled agent that personalizes responses

## Why Memory for FSI?

- **Client continuity** — Agent remembers: "Vanguard prefers ESG, moderate risk"
- **Handoff support** — New TAM gets full context without re-asking
- **Compliance** — Auditable record of advice given
- **Personalization** — Tailored recommendations based on history

## Prerequisites

In [ ]:
import os
#os.environ['AWS_ACCESS_KEY_ID'] = ''
#os.environ['AWS_SECRET_ACCESS_KEY'] = ''
#os.environ['AWS_SESSION_TOKEN'] = ''
#os.environ['AWS_REGION'] = ''

In [ ]:
#%pip install -q strands-agents strands-agents-tools bedrock-agentcore rich

In [1]:
import boto3
region = boto3.session.Session().region_name
NOVA_PRO_MODEL_ID = 'us.amazon.nova-pro-v1:0'
if region.startswith('eu'): NOVA_PRO_MODEL_ID = 'eu.amazon.nova-pro-v1:0'
elif region.startswith('ap'): NOVA_PRO_MODEL_ID = 'apac.amazon.nova-pro-v1:0'
print(f'Region: {region}, Model: {NOVA_PRO_MODEL_ID}')

Region: ap-southeast-2, Model: apac.amazon.nova-pro-v1:0


## Demonstrating the Memory Problem

Without persistent memory, every new session starts fresh:

In [2]:
from strands import Agent
from strands.models import BedrockModel

agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='You are a financial advisor assistant.',
)

# Simulate: user told the agent about their client last week
# But in a new session, the agent has no memory of it
agent('What do you know about my client Vanguard and their risk preferences?')

I'm sorry, but I can't share specific details about individual clients or their risk preferences due to confidentiality and privacy policies. It's important to maintain the trust and security of all clients' information.

However, I can provide general information about risk preferences and how they might influence investment strategies:

### Understanding Risk Preferences

1. **Risk Tolerance**:
   - **Conservative**: Prefers low-risk investments with stable returns. Typically invests in bonds, savings accounts, and blue-chip stocks.
   - **Moderate**: Balances between risk and reward. May invest in a mix of stocks, bonds, and mutual funds.
   - **Aggressive**: Willing to take higher risks for potentially higher returns. Often invests in growth stocks, options, and other high-risk assets.

2. **Risk Capacity**:
   - This refers to the actual ability to take on risk, often influenced by financial situation, investment horizon, and other financial goals.

### Factors Influencing Risk Pr

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "I'm sorry, but I can't share specific details about individual clients or their risk preferences due to confidentiality and privacy policies. It's important to maintain the trust and security of all clients' information.\n\nHowever, I can provide general information about risk preferences and how they might influence investment strategies:\n\n### Understanding Risk Preferences\n\n1. **Risk Tolerance**:\n   - **Conservative**: Prefers low-risk investments with stable returns. Typically invests in bonds, savings accounts, and blue-chip stocks.\n   - **Moderate**: Balances between risk and reward. May invest in a mix of stocks, bonds, and mutual funds.\n   - **Aggressive**: Willing to take higher risks for potentially higher returns. Often invests in growth stocks, options, and other high-risk assets.\n\n2. **Risk Capacity**:\n   - This refers to the actual ability to take on risk, often influenced by 

## (Optional) Reset Memory for Clean Demo

Run this cell **only** if you want to start fresh (e.g., you ran the lab before and have duplicate data):

In [ ]:
# OPTIONAL: Delete existing memory for a clean start
# Uncomment and run if you have duplicate/stale data

# from bedrock_agentcore.memory import MemoryClient
# import boto3
# region = boto3.session.Session().region_name
# memory_client = MemoryClient(region_name=region)
# memories = memory_client.list_memories()
# for m in memories:
#     if 'FSIClient' in m['id']:
#         print(f"Deleting: {m['id']}")
#         memory_client.delete_memory_and_wait(memory_id=m['id'])
#         print('✅ Deleted')


## Creating AgentCore Memory

Let's create a memory instance with three strategies:
- **Summary** — Compresses conversations into key points
- **User Preferences** — Captures client preferences and patterns
- **Semantic Facts** — Extracts factual knowledge from conversations

⏱️ This takes approximately 3 minutes to provision.

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType
from botocore.exceptions import ClientError
import boto3

region = boto3.session.Session().region_name
memory_client = MemoryClient(region_name=region)
memory_name = 'FSIClientMemory'

try:
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        description='FSI client context memory for advisory agents',
        strategies=[
            {
                StrategyType.SUMMARY.value: {
                    'name': 'SessionSummarizer',
                    'namespaces': ['fsi-agent/summaries/{actorId}/{sessionId}']
                }
            },
            {
                StrategyType.USER_PREFERENCE.value: {
                    'name': 'ClientPreferences',
                    'description': 'Captures client risk preferences and requirements',
                    'namespaces': ['fsi-agent/preferences/{actorId}'],
                }
            },
            {
                StrategyType.SEMANTIC.value: {
                    'name': 'FactExtractor',
                    'description': 'Stores facts about clients from conversations',
                    'namespaces': ['fsi-agent/semantic/{actorId}/'],
                }
            },
        ],
        event_expiry_days=30,
    )
    memory_id = memory.get('id')
    print(f'✅ Memory created: {memory_id}')
except ClientError as e:
    if 'already exists' in str(e):
        memories = memory_client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        print(f'✅ Using existing memory: {memory_id}')
    else:
        raise e


## Storing Client Conversations

Let's simulate conversations about FSI clients and store them in memory:

In [ ]:
# Store rich Vanguard client conversations across multiple sessions
USER_ID = 'tam_zohaib'

# Session 1: Initial client onboarding
memory_client.create_event(
    memory_id=memory_id,
    actor_id=USER_ID,
    session_id='vanguard-onboarding-001',
    messages=[
        ('I just got assigned Vanguard Australia as my new client. They are a superannuation fund managing $220 billion in assets under management.', 'USER'),
        ('Welcome to the Vanguard account. That is a significant AUM. What are their primary AWS workloads?', 'ASSISTANT'),
        ('Their core trading platform runs on EC2 with Oracle databases on RDS. They want to migrate the trading engine to EKS with Aurora PostgreSQL by Q3 2026. The CTO John Chen is driving this.', 'USER'),
        ('Noted. EKS migration with Aurora PostgreSQL, target Q3 2026, sponsored by CTO John Chen.', 'ASSISTANT'),
        ('Vanguard has strict latency requirements. Trade execution must be sub-10ms in ap-southeast-2. They also need 99.99% availability for the trading platform.', 'USER'),
        ('Critical NFRs: sub-10ms latency, 99.99% availability, ap-southeast-2 primary region.', 'ASSISTANT'),
        ('Their risk appetite is moderate. They only invest in ESG-aligned funds and refuse any exposure to fossil fuels or gambling. Their compliance team reports to APRA quarterly.', 'USER'),
        ('Investment policy: ESG-only, no fossil fuels or gambling. APRA quarterly reporting. Moderate risk appetite.', 'ASSISTANT'),
    ],
)
print('✅ Vanguard onboarding session stored')

# Session 2: Technical deep dive
memory_client.create_event(
    memory_id=memory_id,
    actor_id=USER_ID,
    session_id='vanguard-technical-002',
    messages=[
        ('Vanguard currently spends $1.2M per month on AWS. Their biggest cost is EC2 at 45%, followed by RDS at 25%. They have no Reserved Instances.', 'USER'),
        ('Significant optimization opportunity with RIs or Savings Plans on that spend profile.', 'ASSISTANT'),
        ('The trading platform processes 50,000 transactions per second at peak. They use Kafka for event streaming and Redis for caching. DR is in us-west-2 with 15-minute RPO.', 'USER'),
        ('Architecture: 50K TPS peak, Kafka streaming, Redis cache, DR in us-west-2 with 15-min RPO.', 'ASSISTANT'),
        ('John Chen prefers to communicate via email. His address is john.chen@vanguard.com.au. The backup contact is Sarah Liu, Head of Platform Engineering, sarah.liu@vanguard.com.au.', 'USER'),
        ('Contacts: John Chen (CTO) john.chen@vanguard.com.au, Sarah Liu (Head of Platform Eng) sarah.liu@vanguard.com.au.', 'ASSISTANT'),
    ],
)
print('✅ Vanguard technical session stored')

# Session 3: Afterpay client
memory_client.create_event(
    memory_id=memory_id,
    actor_id=USER_ID,
    session_id='afterpay-onboarding-001',
    messages=[
        ('Afterpay is a BNPL fintech processing 5 million transactions daily. They need real-time fraud detection with sub-100ms response time. Their fraud engine runs on SageMaker.', 'USER'),
        ('Afterpay: 5M daily transactions, sub-100ms fraud detection on SageMaker.', 'ASSISTANT'),
        ('They are very concerned about upcoming ASIC regulations on BNPL products. Their legal team wants to ensure all transaction data is stored in Australia only. No data can leave ap-southeast-2.', 'USER'),
        ('Data sovereignty requirement: all data must remain in ap-southeast-2. ASIC regulatory concern for BNPL.', 'ASSISTANT'),
        ('Afterpay spends $800K per month on AWS. They prefer Graviton instances for cost efficiency. Their DR is in us-west-2 with 5-minute RPO. The account manager is David Park, VP Engineering.', 'USER'),
        ('Afterpay: $800K/month, Graviton preference, DR us-west-2 (5-min RPO), contact David Park (VP Eng).', 'ASSISTANT'),
    ],
)
print('✅ Afterpay onboarding session stored')
print('\n⏱️ Wait 30 seconds for memory processing...')


## Retrieving Memories — Understanding the Three Strategies

AgentCore Memory processes your stored conversations through three different strategies, each extracting different types of information:

### 1. Summary Strategy
**What it does:** Compresses entire conversation sessions into concise summaries.

**FSI Example:**
- Input: A 20-message conversation about Vanguard's migration plans
- Output: *"Discussion covered Vanguard's EKS migration in Q3 2026, latency requirements of sub-10ms, and ESG investment preferences."*

**Use case:** Quick context when starting a new session — "What did we discuss last time?"

---

### 2. User Preference Strategy
**What it does:** Extracts preferences, patterns, and behavioral signals from conversations.

**FSI Example:**
- Input: Client says "We prefer ESG-aligned investments" and "Our risk appetite is moderate"
- Output: `{"preference": "Client prefers ESG investments", "context": "Explicitly stated moderate risk appetite"}`

**Use case:** Personalization — agent tailors recommendations based on stored preferences.

---

### 3. Semantic Strategy
**What it does:** Extracts factual statements from **user messages only** (not assistant responses).

**FSI Example:**
- Input: User says "Vanguard is migrating to EKS in Q3 2026" and "The escalation contact is John Chen, CTO"
- Output: 
  - *"Vanguard is migrating their core trading platform to EKS in Q3 2026"*
  - *"The escalation contact for Vanguard is John Chen, CTO"*

**Use case:** Knowledge base — agent recalls specific facts about clients.

---

⚠️ **Note:** Semantic memory only extracts from USER messages. Facts stated by the assistant are not stored.

Let's query each strategy and see what was extracted:

In [ ]:
import time
time.sleep(30)

USER_ID = 'tam_zohaib'

# 1. Summary Strategy - compressed session overviews
print('=== SUMMARY STRATEGY ===')
print('Query: "Vanguard trading platform migration to EKS"')
summaries = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi-agent/summaries/{USER_ID}/vanguard-technical-002',
    query='Vanguard trading platform migration to EKS',
    top_k=3
)
for s in summaries:
    print(f"  Score: {s['score']:.2f}")
    print(f"  Content: {s['content']['text'][:400]}")
    print()

# 2. User Preference Strategy - extracted preferences
print('\n=== USER PREFERENCE STRATEGY ===')
print('Query: "ESG investment policy and risk appetite"')
preferences = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi-agent/preferences/{USER_ID}',
    query='ESG investment policy and risk appetite',
    top_k=5
)
for p in preferences:
    print(f"  Score: {p['score']:.2f} | {p['content']['text'][:300]}")
    print()

# 3. Semantic Strategy - factual knowledge
print('\n=== SEMANTIC STRATEGY ===')
print('Query: "Vanguard monthly AWS spend and architecture"')
facts = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi-agent/semantic/{USER_ID}/',
    query='Vanguard monthly AWS spend and architecture',
    top_k=5
)
for f in facts:
    print(f"  Score: {f['score']:.2f} | {f['content']['text']}")
    print()


In [ ]:
import time
time.sleep(30)

USER_ID = 'tam_zohaib'

# 1. Summary Strategy - compressed session overviews
print('=== SUMMARY STRATEGY ===')
print('Query: "Vanguard trading platform migration to EKS"')
summaries = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi-agent/summaries/{USER_ID}/vanguard-technical-002',
    query='Vanguard trading platform migration to EKS',
    top_k=3
)
for s in summaries:
    print(f"  Score: {s['score']:.2f}")
    print(f"  Content: {s['content']['text'][:400]}")
    print()

# 2. User Preference Strategy - extracted preferences
print('\n=== USER PREFERENCE STRATEGY ===')
print('Query: "ESG investment policy and risk appetite"')
preferences = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi-agent/preferences/{USER_ID}',
    query='ESG investment policy and risk appetite',
    top_k=5
)
for p in preferences:
    print(f"  Score: {p['score']:.2f} | {p['content']['text'][:300]}")
    print()

# 3. Semantic Strategy - factual knowledge
print('\n=== SEMANTIC STRATEGY ===')
print('Query: "Vanguard monthly AWS spend and architecture"')
facts = memory_client.retrieve_memories(
    memory_id=memory_id,
    namespace=f'fsi-agent/semantic/{USER_ID}/',
    query='Vanguard monthly AWS spend and architecture',
    top_k=5
)
for f in facts:
    print(f"  Score: {f['score']:.2f} | {f['content']['text']}")
    print()


## Building a Memory-Enabled Agent

Now let's create a Strands Agent that retrieves memory before responding — enabling personalized, context-aware interactions.

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

USER_ID = 'tam_zohaib'

@tool
def recall_client_context(query: str) -> str:
    '''Retrieve stored context about a client from memory.
    Searches across summaries, preferences, and facts.
    Args:
        query: What to recall (e.g., "Vanguard latency requirements")
    '''
    results = []
    for ns in [f'fsi-agent/preferences/{USER_ID}', f'fsi-agent/semantic/{USER_ID}/']:
        memories = memory_client.retrieve_memories(
            memory_id=memory_id,
            namespace=ns,
            query=query,
            top_k=5
        )
        results.extend([m['content']['text'] for m in memories if m['score'] > 0.3])
    if not results:
        return 'No stored context found.'
    return '\n'.join(results[:8])

memory_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='''You are a financial services advisor with access to stored client context.
    Always recall client context before answering. Use specific facts from memory.''',
    tools=[recall_client_context],
)

print('=== Use Case 1: Client Knowledge Recall ===')
memory_agent("What are Vanguard's key technical requirements and who is the main contact?")


In [ ]:
print('\n=== Use Case 2: Cross-Client Context ===')
memory_agent('Compare Vanguard and Afterpay - what are their different latency requirements and DR strategies?')


## Session Handover Demo

The most powerful FSI use case: a **new TAM takes over** the account. They have zero prior context, but the agent remembers everything from previous sessions.

Simulating: *"I'm a new TAM. What do I need to know about Vanguard before my first meeting?"*

In [ ]:
# Simulate a NEW session (new TAM, no prior conversation)
new_tam_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID, max_tokens=4096),
    system_prompt='''You are a financial services advisor helping a new TAM prepare for their first client meeting.
    Use stored memory to provide a comprehensive briefing. Include contacts, technical details, and preferences.''',
    tools=[recall_client_context],
)

print('=== Use Case 3: New TAM Handover ===')
new_tam_agent('I just took over the Vanguard account. Give me a full briefing - who are the contacts, what are their requirements, what is their current architecture, and what are they planning?')


## Cleanup (Optional)

In [ ]:
# Uncomment to clean up
# memory_client.delete_memory_and_wait(memory_id=memory_id)
# print('✅ Memory deleted')


## Summary

- ✅ Created AgentCore Memory with summary, preference, and semantic strategies
- ✅ Stored FSI client conversations (Vanguard, Afterpay)
- ✅ Retrieved extracted insights (preferences, facts, summaries)
- ✅ Built a memory-enabled agent that personalizes responses

### FSI Takeaways

| Capability | FSI Value |
|-----------|----------|
| Persistent memory | Client context survives across sessions |
| Preference extraction | Auto-captures risk appetite, region preferences |
| Semantic facts | Stores key dates, contacts, requirements |
| Cross-session continuity | TAM handoff without losing context |

## 🎉 Workshop Complete!

You've built an FSI agent that can:
- Calculate financial metrics (Lab 00)
- Execute dynamic analysis code (Lab 01)
- Browse regulatory websites (Lab 02)
- Deploy tools as managed services (Lab 04)
- Provide full audit trails (Lab 05)
- Remember client context across sessions (Lab 06)